# Zep
> 回忆、理解并从聊天记录中提取数据。赋能个性化 AI 体验。

> [Zep](https://www.getzep.com) 是 AI 助手应用的长期记忆服务。
> 使用 Zep，您可以让 AI 助手回忆起过去的对话，无论多么久远，
> 同时还能减少幻觉、延迟和成本。

> 对 Zep Cloud 感兴趣？请参阅 [Zep Cloud 安装指南](https://help.getzep.com/sdks) 和 [Zep Cloud 向量存储示例](https://help.getzep.com/langchain/examples/vectorstore-example)

## 开源安装与设置

> Zep 开源项目：[https://github.com/getzep/zep](https://github.com/getzep/zep)
>
> Zep 开源文档：[https://docs.getzep.com/](https://docs.getzep.com/)

您需要安装 `langchain-community`，命令为 `pip install -qU langchain-community` 才能使用此集成。

## 用法

在下面的示例中，我们使用了 Zep 的自动嵌入功能，该功能会在 Zep 服务器上使用低延迟嵌入模型自动嵌入文档。

## 注意
- 这些示例使用了 Zep 的异步接口。通过移除方法名称中的 `a` 前缀即可调用同步接口。
- 如果您传入一个 `Embeddings` 实例，Zep 将使用它来嵌入文档，而不是自动嵌入。
您还必须将您的文档集合设置为 `isAutoEmbedded === false`。
- 如果您将集合设置为 `isAutoEmbedded === false`，则必须传入一个 `Embeddings` 实例。

## 从文档加载或创建 Collection

In [1]:
from uuid import uuid4

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import ZepVectorStore
from langchain_community.vectorstores.zep import CollectionConfig
from langchain_text_splitters import RecursiveCharacterTextSplitter

ZEP_API_URL = "http://localhost:8000"  # this is the API url of your Zep instance
ZEP_API_KEY = "<optional_key>"  # optional API Key for your Zep instance
collection_name = f"babbage{uuid4().hex}"  # a unique collection name. alphanum only

# Collection config is needed if we're creating a new Zep Collection
config = CollectionConfig(
    name=collection_name,
    description="<optional description>",
    metadata={"optional_metadata": "associated with the collection"},
    is_auto_embedded=True,  # we'll have Zep embed our documents using its low-latency embedder
    embedding_dimensions=1536,  # this should match the model you've configured Zep to use.
)

# load the document
article_url = "https://www.gutenberg.org/cache/epub/71292/pg71292.txt"
loader = WebBaseLoader(article_url)
documents = loader.load()

# split it into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
docs = text_splitter.split_documents(documents)

# Instantiate the VectorStore. Since the collection does not already exist in Zep,
# it will be created and populated with the documents we pass in.
vs = ZepVectorStore.from_documents(
    docs,
    collection_name=collection_name,
    config=config,
    api_url=ZEP_API_URL,
    api_key=ZEP_API_KEY,
    embedding=None,  # we'll have Zep embed our documents using its low-latency embedder
)

In [2]:
# wait for the collection embedding to complete


async def wait_for_ready(collection_name: str) -> None:
    import time

    from zep_python import ZepClient

    client = ZepClient(ZEP_API_URL, ZEP_API_KEY)

    while True:
        c = await client.document.aget_collection(collection_name)
        print(
            "Embedding status: "
            f"{c.document_embedded_count}/{c.document_count} documents embedded"
        )
        time.sleep(1)
        if c.status == "ready":
            break


await wait_for_ready(collection_name)

Embedding status: 0/401 documents embedded
Embedding status: 0/401 documents embedded
Embedding status: 0/401 documents embedded
Embedding status: 0/401 documents embedded
Embedding status: 0/401 documents embedded
Embedding status: 0/401 documents embedded
Embedding status: 401/401 documents embedded


## 集合上的相似性搜索查询

The similarity search is a common operation in many applications, such as recommendation systems, image retrieval, and natural language processing. It aims to find items in a collection that are similar to a given query item.

相似性搜索是许多应用中的常见操作，例如推荐系统、图像检索和自然语言处理。它旨在在集合中查找与给定查询项相似的项目。

In this section, we will discuss how to perform similarity search queries over a collection of data.

在本节中，我们将讨论如何在数据集合上执行相似性搜索查询。

### 1. Define the Similarity Metric

### 1. 定义相似性度量

The first step in performing a similarity search is to define a similarity metric that quantifies the degree of similarity between two items. The choice of the similarity metric depends on the type of data and the specific application. Some common similarity metrics include:

执行相似性搜索的第一步是定义一个相似性度量，该度量量化两个项目之间的相似程度。相似性度量的选择取决于数据的类型和具体应用。一些常见的相似性度量包括：

*   **Cosine Similarity:** Commonly used for text data, it measures the cosine of the angle between two non-zero vectors. A higher cosine value indicates greater similarity.

    *   **余弦相似度：** 常用于文本数据，它衡量两个非零向量之间夹角的余弦值。较高的余弦值表示较高的相似度。

*   **Euclidean Distance:** Measures the straight-line distance between two points in a multi-dimensional space. Smaller distances indicate greater similarity.

    *   **欧氏距离：** 衡量多维空间中两点之间的直线距离。较小的距离表示较高的相似度。

*   **Jaccard Similarity:** Used for comparing sets, it measures the size of the intersection divided by the size of the union of two sets. A higher Jaccard index indicates greater similarity.

    *   **杰卡德相似度：** 用于比较集合，它衡量两个集合的交集大小除以并集大小。较高的杰卡德指数表示较高的相似度。

### 2. Indexing the Collection

### 2. 索引集合

To efficiently perform similarity search, it is crucial to index the collection of data. Indexing allows us to quickly retrieve items that are likely to be similar to the query item, without having to compare the query item with every item in the collection. Various indexing techniques can be used, depending on the similarity metric and the data distribution. Some popular indexing methods include:

为了高效地执行相似性搜索，对数据集合进行索引至关重要。索引使我们能够快速检索可能与查询项相似的项目，而无需将查询项与集合中的每个项目进行比较。可以根据相似性度量和数据分布使用各种索引技术。一些流行的索引方法包括：

*   **k-d Trees:** Suitable for low-dimensional data, k-d trees partition the space recursively.

    *   **k-d 树：** 适用于低维数据，k-d 树递归地划分空间。

*   **Locality-Sensitive Hashing (LSH):** A probabilistic technique that hashes similar items into the same "buckets" with high probability. It is effective for high-dimensional data.

    *   **局部敏感哈希 (LSH)：** 一种概率技术，以高概率将相似的项目哈希到相同的“桶”中。它对高维数据有效。

*   **Inverted File Index (IVF):** Often used with quantization techniques (like Product Quantization), IVF clusters data points and stores an index for each cluster.

    *   **倒排文件索引 (IVF)：** 通常与量化技术（如乘积量化）一起使用，IVF 对数据点进行聚类，并为每个聚类存储一个索引。

### 3. Performing the Similarity Search Query

### 3. 执行相似性搜索查询

Once the collection is indexed, you can perform a similarity search query. The process typically involves:

一旦集合被索引，您就可以执行相似性搜索查询。该过程通常包括：

1.  **Represent the query item:** Convert the query item into the same format as the items in the collection (e.g., a vector).

    1.  **表示查询项：** 将查询项转换为与集合中的项目相同的格式（例如，向量）。

2.  **Query the index:** Use the index to find candidate items that are likely to be similar to the query item. The specific query method depends on the indexing technique used. For example, with LSH, you would hash the query item and retrieve items from the corresponding buckets.

    2.  **查询索引：** 使用索引查找可能与查询项相似的候选项目。具体的查询方法取决于使用的索引技术。例如，使用 LSH，您将对查询项进行哈希处理，并从相应的桶中检索项目。

3.  **Rank the results:** Calculate the similarity score between the query item and the candidate items using the defined similarity metric. Sort the results in descending order of similarity.

    3.  **对结果进行排序：** 使用定义的相似性度量计算查询项与候选项目之间的相似度得分。按相似度降序对结果进行排序。

4.  **Return the top-k results:** Return the top `k` most similar items to the user.

    4.  **返回 top-k 结果：** 向用户返回 `k` 个最相似的项目。

**Example using a hypothetical library:**

**使用假设库的示例：**

```python
from some_similarity_library import Collection, CosineSimilarity, KDTreeIndex

# Assume 'data' is a list of items (e.g., documents, images)
# Assume 'item_to_vector' is a function that converts an item to its vector representation

# 1. Define the similarity metric
similarity_metric = CosineSimilarity()

# 2. Index the collection
# Convert items to vectors before indexing
vectors = [item_to_vector(item) for item in data]
index = KDTreeIndex(vectors)
collection = Collection(data, index, similarity_metric)

# 3. Perform the similarity search query
query_item = get_query_item()
query_vector = item_to_vector(query_item)
top_k_results = collection.search(query_vector, k=5)

print("Top 5 similar items:")
for item, score in top_k_results:
    print(f"- {item}: {score:.4f}")
```

```python
from some_similarity_library import Collection, CosineSimilarity, KDTreeIndex

# 假设 'data' 是项目列表（例如，文档、图像）
# 假设 'item_to_vector' 是一个将项目转换为其向量表示的函数

# 1. 定义相似性度量
similarity_metric = CosineSimilarity()

# 2. 索引集合
# 在索引之前将项目转换为向量
vectors = [item_to_vector(item) for item in data]
index = KDTreeIndex(vectors)
collection = Collection(data, index, similarity_metric)

# 3. 执行相似性搜索查询
query_item = get_query_item()
query_vector = item_to_vector(query_item)
top_k_results = collection.search(query_vector, k=5)

print("最相似的 5 个项目：")
for item, score in top_k_results:
    print(f"- {item}: {score:.4f}")
```

This example demonstrates a basic workflow for similarity search. The specific implementation details will vary depending on the chosen library and techniques.

这个例子展示了相似性搜索的基本工作流程。具体的实现细节将根据所选的库和技术而有所不同。

In [3]:
# query it
query = "what is the structure of our solar system?"
docs_scores = await vs.asimilarity_search_with_relevance_scores(query, k=3)

# print results
for d, s in docs_scores:
    print(d.page_content, " -> ", s, "\n====\n")

the positions of the two principal planets, (and these the most
necessary for the navigator,) Jupiter and Saturn, require each not less
than one hundred and sixteen tables. Yet it is not only necessary to
predict the position of these bodies, but it is likewise expedient to
tabulate the motions of the four satellites of Jupiter, to predict the
exact times at which they enter his shadow, and at which their shadows
cross his disc, as well as the times at which they are interposed  ->  0.9003241539387915 
====

furnish more than a small fraction of that aid to navigation (in the
large sense of that term), which, with greater facility, expedition, and
economy in the calculation and printing of tables, it might be made to
supply.

Tables necessary to determine the places of the planets are not less
necessary than those for the sun, moon, and stars. Some notion of the
number and complexity of these tables may be formed, when we state that  ->  0.8911165633479508 
====

the scheme of notation

## 通过 MMR 对集合搜索进行重新排名

Zep 提供原生、硬件加速的 MMR 重新排名搜索结果的功能。

In [4]:
query = "what is the structure of our solar system?"
docs = await vs.asearch(query, search_type="mmr", k=3)

for d in docs:
    print(d.page_content, "\n====\n")

the positions of the two principal planets, (and these the most
necessary for the navigator,) Jupiter and Saturn, require each not less
than one hundred and sixteen tables. Yet it is not only necessary to
predict the position of these bodies, but it is likewise expedient to
tabulate the motions of the four satellites of Jupiter, to predict the
exact times at which they enter his shadow, and at which their shadows
cross his disc, as well as the times at which they are interposed 
====

the scheme of notation thus applied, immediately suggested the
advantages which must attend it as an instrument for expressing the
structure, operation, and circulation of the animal system; and we
entertain no doubt of its adequacy for that purpose. Not only the
mechanical connexion of the solid members of the bodies of men and
animals, but likewise the structure and operation of the softer parts,
including the muscles, integuments, membranes, &c. the nature, motion, 
====

resistance, economizing time, 

# 按元数据筛选

使用元数据过滤器来缩小搜索范围。首先，加载另一本书：“Adventures of Sherlock Holmes”

In [5]:
# Let's add more content to the existing Collection
article_url = "https://www.gutenberg.org/files/48320/48320-0.txt"
loader = WebBaseLoader(article_url)
documents = loader.load()

# split it into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
docs = text_splitter.split_documents(documents)

await vs.aadd_documents(docs)

await wait_for_ready(collection_name)

Embedding status: 401/1691 documents embedded
Embedding status: 401/1691 documents embedded
Embedding status: 401/1691 documents embedded
Embedding status: 401/1691 documents embedded
Embedding status: 401/1691 documents embedded
Embedding status: 401/1691 documents embedded
Embedding status: 901/1691 documents embedded
Embedding status: 901/1691 documents embedded
Embedding status: 901/1691 documents embedded
Embedding status: 901/1691 documents embedded
Embedding status: 901/1691 documents embedded
Embedding status: 901/1691 documents embedded
Embedding status: 1401/1691 documents embedded
Embedding status: 1401/1691 documents embedded
Embedding status: 1401/1691 documents embedded
Embedding status: 1401/1691 documents embedded
Embedding status: 1691/1691 documents embedded


我们看到了两本书的结果。请注意 `source` 元数据

In [6]:
query = "Was he interested in astronomy?"
docs = await vs.asearch(query, search_type="similarity", k=3)

for d in docs:
    print(d.page_content, " -> ", d.metadata, "\n====\n")

or remotely, for this purpose. But in addition to these, a great number
of tables, exclusively astronomical, are likewise indispensable. The
predictions of the astronomer, with respect to the positions and motions
of the bodies of the firmament, are the means, and the only means, which
enable the mariner to prosecute his art. By these he is enabled to
discover the distance of his ship from the Line, and the extent of his  ->  {'source': 'https://www.gutenberg.org/cache/epub/71292/pg71292.txt'} 
====

possess all knowledge which is likely to be useful to him in his work,
and this I have endeavored in my case to do. If I remember rightly, you
on one occasion, in the early days of our friendship, defined my limits
in a very precise fashion.”

“Yes,” I answered, laughing. “It was a singular document. Philosophy,
astronomy, and politics were marked at zero, I remember. Botany
variable, geology profound as regards the mud-stains from any region  ->  {'source': 'https://www.gutenberg.org/file

现在，我们设置一个过滤器

In [7]:
filter = {
    "where": {
        "jsonpath": (
            "$[*] ? (@.source == 'https://www.gutenberg.org/files/48320/48320-0.txt')"
        )
    },
}

docs = await vs.asearch(query, search_type="similarity", metadata=filter, k=3)

for d in docs:
    print(d.page_content, " -> ", d.metadata, "\n====\n")

possess all knowledge which is likely to be useful to him in his work,
and this I have endeavored in my case to do. If I remember rightly, you
on one occasion, in the early days of our friendship, defined my limits
in a very precise fashion.”

“Yes,” I answered, laughing. “It was a singular document. Philosophy,
astronomy, and politics were marked at zero, I remember. Botany
variable, geology profound as regards the mud-stains from any region  ->  {'source': 'https://www.gutenberg.org/files/48320/48320-0.txt'} 
====

the light shining upon his strong-set aquiline features. So he sat as I
dropped off to sleep, and so he sat when a sudden ejaculation caused me
to wake up, and I found the summer sun shining into the apartment. The
pipe was still between his lips, the smoke still curled upward, and the
room was full of a dense tobacco haze, but nothing remained of the heap
of shag which I had seen upon the previous night.

“Awake, Watson?” he asked.

“Yes.”

“Game for a morning drive?”  ->